# Estimating Shapley Value Feature Attributions

This notebook turns the taxonomy of Shapley-value estimators (Chen, Covert, Lundberg & Lee, 2023 and related work) into **hands-on experiments**. Each module makes one specific conceptual point tangible with runnable code, rather than leaving it abstract.


## Setup
Install the packages we'll use. This may take a minute the first time.

In [ ]:
# Core scientific stack is preinstalled in Colab; we only need to add SHAP-related libraries.
!pip install -q shap shapiq xgboost


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.5/69.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.9/17.9 MB 33.6 MB/s eta 0:00:00


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from itertools import combinations
import math
import time

rng_global = np.random.default_rng(0)
np.set_printoptions(precision=4, suppress=True)


## **Exercise 1 : Exact vs. sampled Shapley values**

We build a small **toy cooperative game** over $n=8$ "players" (features), small enough to
brute-force enumerate all $2^n$ coalitions and get the *exact* Shapley value. We then compare
two sampling-based estimators from the semivalue / random-order family:

- **ApproSemivalue** (Castro, Gómez & Tejada, 2009) — samples a random subset per player, independently.
- **ApproShapley** (Castro, Gómez & Tejada, 2009) — samples a random *permutation*, and updates
  every player's estimate from the same walk (query reuse).

The point of this module: both are unbiased estimators of the *same* quantity, but they behave
differently as a function of the sampling budget.


In [ ]:
# --- Toy cooperative game with mild synergy (not purely additive) ---
n = 8
weights = rng_global.uniform(0.2, 1.0, size=n)

def v(S, weights=weights):
    """Coalitional game value for subset S (a set or list of player indices)."""
    S = tuple(sorted(S))
    if len(S) == 0:
        return 0.0
    w = weights[list(S)].sum()
    synergy = 0.15 * sum(weights[i] * weights[j] for i, j in combinations(S, 2))
    return w + synergy

players = list(range(n))
print("Toy game with", n, "players. v(full coalition) =", round(v(players), 3))


Toy game with 8 players. v(full coalition) = 6.306


In [ ]:
# --- Exact Shapley value via brute-force enumeration (feasible only because n is small) ---
def exact_shapley(players, v):
    n = len(players)
    phi = np.zeros(n)
    for i in players:
        others = [p for p in players if p != i]
        total = 0.0
        for r in range(len(others) + 1):
            for S in combinations(others, r):
                weight = (math.factorial(r) * math.factorial(n - r - 1)) / math.factorial(n)
                total += weight * (v(set(S) | {i}) - v(set(S)))
        phi[i] = total
    return phi

phi_exact = exact_shapley(players, v)
print("Exact Shapley values:", phi_exact)
print("Sum of attributions:  ", round(phi_exact.sum(), 4))
print("v(N) - v(empty):      ", round(v(players) - v([]), 4))
print("(these two should match exactly -- this IS the efficiency axiom)")


Exact Shapley values: [0.9284 0.5532 0.3129 0.2869 1.1039 1.2017 0.8979 1.0209]
Sum of attributions:   6.3057
v(N) - v(empty):       6.3057
(these two should match exactly -- this IS the efficiency axiom)


## **Implementation of ApproSemivalue and ApproShapley and test**

In [ ]:
# --- ApproSemivalue: independent subset sampling per player ---
def approsemivalue(players, v, m_per_player, rng):
    n = len(players)
    phi = np.zeros(n)
    for i in players:
        others = [p for p in players if p != i]
        contribs = []
        for _ in range(m_per_player):
            r = rng.integers(0, len(others) + 1)
            S = set(rng.choice(others, size=r, replace=False)) if r > 0 else set()
            contribs.append(v(S | {i}) - v(S))
        phi[i] = np.mean(contribs)
    return phi


In [ ]:
# --- ApproShapley: permutation sampling, ALL players updated per sampled permutation ---
def approshapley(players, v, num_perms, rng):
    n = len(players)
    phi = np.zeros(n)
    for _ in range(num_perms):
        perm = rng.permutation(players)
        S = set()
        prev_value = v(S)
        for p in perm:
            new_value = v(S | {p})
            phi[p] += (new_value - prev_value)
            S = S | {p}
            prev_value = new_value
    return phi / num_perms

In [ ]:
# Quick sanity check at a moderate budget
rng1 = np.random.default_rng(1)
phi_sv_demo = approsemivalue(players, v, 300, rng1)
phi_ro_demo = approshapley(players, v, 300, rng1)
print("SV  estimate error (L2):", np.linalg.norm(phi_sv_demo - phi_exact))
print("RO  estimate error (L2):", np.linalg.norm(phi_ro_demo - phi_exact))

SV  estimate error (L2): 0.018861254654415675
RO  estimate error (L2): 0.025281502396773424


## **Question 1 : compare the convergence rate of these two estimators and comment**

## **Question 2 : the efficiency axiom**

The exact Shapley value always satisfies $\sum_i \phi_i = v(N) - v(\emptyset)$ (the *efficiency*
or *local accuracy* axiom). Random-order (permutation) sampling satisfies this **exactly**, even
from a single sampled permutation, because the marginal contributions along one permutation
telescope to the total. Semivalue sampling only satisfies it **in expectation**, as the sample
size grows.

Check it!

## **Exercise 2 : KernelSHAP as weighted least squares, from scratch**


KernelSHAP (Lundberg & Lee, 2017) reformulates the Shapley value as the solution to a
**weighted least-squares regression** over sampled coalitions. We implement this by hand on our
toy game, then validate it against the official `shap` library on a real linear model (where
we can also compare to the *exact* linear SHAP formula).


In [ ]:
def kernelshap_manual(players, v, num_samples, rng):
    """KernelSHAP built from scratch: sample coalitions, solve a weighted least-squares
    regression using the Shapley kernel, and enforce the efficiency constraint exactly
    (the standard reduced-regression trick)."""
    n = len(players)
    v_empty = v(set())
    v_full = v(set(players))

    rows, targets, weights = [], [], []
    for _ in range(num_samples):
        size = rng.integers(1, n)  # exclude empty/full coalitions (infinite kernel weight)
        S = set(rng.choice(players, size=size, replace=False))
        z = np.zeros(n)
        z[list(S)] = 1
        s = len(S)
        w = (n - 1) / (math.comb(n, s) * s * (n - s))  # Shapley kernel weight W(S)
        rows.append(z); targets.append(v(S)); weights.append(w)

    Z = np.array(rows); y = np.array(targets); W = np.array(weights)

    # Enforce efficiency constraint by reducing to n-1 free variables (standard trick)
    y_adj = y - v_empty
    total = v_full - v_empty
    Z_reduced = Z[:, :-1] - Z[:, -1:].reshape(-1, 1)
    y_reduced = y_adj - Z[:, -1] * total

    WZ = Z_reduced * np.sqrt(W)[:, None]
    Wy = y_reduced * np.sqrt(W)
    beta_reduced, *_ = np.linalg.lstsq(WZ, Wy, rcond=None)

    phi = np.zeros(n)
    phi[:-1] = beta_reduced
    phi[-1] = total - beta_reduced.sum()
    return phi


## **Question 1 : test this estimator and calculate the error. Does the estimator satisfy the efficiency axiom?**

## **Question 2 : Validate against the official `shap` library on a real linear model**

In [ ]:
import shap
from sklearn.linear_model import LinearRegression
from sklearn.datasets import load_diabetes

X, y = load_diabetes(return_X_y=True)
X, y = X[:150, :4], y[:150]   # keep small & fast for the lab
model = LinearRegression().fit(X, y)
background = X[:30]


## **Exercise 3 : Marginal vs. conditional Shapley values with correlated features**


This is the most important conceptual experiment for the **feature-removal axis** of the
taxonomy. We build a dataset with two nearly-identical (highly correlated) features, feeding a
model that only *causally* depends on one of them. We then compute Shapley values two ways:

- **Marginal (interventional)** — "absent" features are replaced by independent draws from a
  background sample (what KernelSHAP and TreeSHAP do by default).
- **Conditional** — "absent" features are replaced by draws that respect the *conditional*
  distribution given the observed features (approximated here with a simple k-nearest-neighbors
  scheme, standing in for methods like Aas, Jullum & Løland's Gaussian/copula estimators).

**Prediction before running:** which feature will each version credit?


In [ ]:
rng4 = np.random.default_rng(42)
n_samples = 2000
x1 = rng4.normal(0, 1, n_samples)
x2 = x1 + rng4.normal(0, 0.05, n_samples)   # near-duplicate of x1 (r ~ 0.999)
x3 = rng4.normal(0, 1, n_samples)           # independent, causally irrelevant
X_corr = np.column_stack([x1, x2, x3])

In [ ]:
def toy_model(X):
    """Model that TRULY depends only on x1 (coefficient 0 on x2), plus a little on x3."""
    return 3 * X[:, 0] + 0 * X[:, 1] + 0.5 * X[:, 2]

In [ ]:
X_bg = X_corr[200:1200]
x_test = X_corr[0]
print("Correlation(x1, x2) =", round(np.corrcoef(x1, x2)[0,1], 4))
print("Point to explain: x_test =", np.round(x_test, 3))


## **Question 1 : what is the correlation between $X_1$ and $X_2$?**

## **Question 2 : define value functions corresponding to marginal game and conditional game**

In [ ]:
# Fix ONE shared background draw for all coalition evaluations: this reduces Monte Carlo
# variance a lot and makes the efficiency axiom hold much more tightly (see Module 2).
FIXED_IDX = np.random.default_rng(7).integers(0, X_bg.shape[0], size=300)
FIXED_BG = X_corr[200:1200][FIXED_IDX]


In [ ]:
def v_marginal(S, x, model):
    """Marginal game: absent features filled in with INDEPENDENT background draws."""
    n = FIXED_BG.shape[1]
    TO FILL


In [ ]:
def v_conditional(S, x, X_bg, model, k=150):
    """Conditional game (approximate): absent features filled in using the k nearest
    background points *given the observed features* -- so correlations are respected."""
    n = X_bg.shape[1]
    TO FILL

## **Question 3 : define Shapley from v**

In [ ]:
def shapley_from_v(v_fn, n=3):
    """Exact Shapley value for an arbitrary black-box value function v_fn (n small)."""
    TO FILL


## **Question 4 : test it**

## **Summary**

In [ ]:
labels = ["x1 (causal)", "x2 (correlated, non-causal)", "x3 (independent)"]
fig, ax = plt.subplots(figsize=(6,4))
width = 0.35
xpos = np.arange(3)
ax.bar(xpos - width/2, phi_marginal, width, label="Marginal Shapley")
ax.bar(xpos + width/2, phi_conditional, width, label="Conditional Shapley")
ax.set_xticks(xpos); ax.set_xticklabels(labels)
ax.set_ylabel("Attribution")
ax.set_title("Marginal vs. conditional Shapley values\nunder feature correlation")
ax.legend(); ax.grid(True, alpha=0.3)
plt.show()


## **Exercise 4 : The "plug-in" pipeline**

Here we operationalize the two-stage decomposition directly: choose a **value-function
estimator** $\hat v(S)$ (how "missing" features are filled in), then choose a **Shapley
estimator** $\hat\phi$ (how marginal contributions are aggregated), and compare all four
combinations on the correlated-feature example from Module 4.


In [ ]:
# Two v-hat estimators, reusing the functions from Module 4
def vhat_mean_impute(S, x, X_bg, model):
    return v_marginal(S, x, model)  # independent background draws (marginal)

def vhat_knn_conditional(S, x, X_bg, model):
    return v_conditional(S, x, X_bg, model)  # nearest-neighbor conditional approximation

In [ ]:
# Two phi-hat estimators: permutation sampling (RO) vs. WLS regression (KernelSHAP-style),
# both adapted to work on an arbitrary black-box v_fn with n=3 players
def phihat_permutation(v_fn, n, num_perms, rng):
    players = list(range(n))
    phi = np.zeros(n)
    for _ in range(num_perms):
        perm = rng.permutation(players)
        S = set(); prev = v_fn(S)
        for p in perm:
            new = v_fn(S | {p})
            phi[p] += new - prev
            S = S | {p}; prev = new
    return phi / num_perms

def phihat_wls(v_fn, n, num_samples, rng):
    v_empty, v_full = v_fn(set()), v_fn(set(range(n)))
    rows, targets, weights = [], [], []
    for _ in range(num_samples):
        size = rng.integers(1, n)
        S = set(rng.choice(n, size=size, replace=False))
        z = np.zeros(n); z[list(S)] = 1
        s = len(S)
        w = (n - 1) / (math.comb(n, s) * s * (n - s))
        rows.append(z); targets.append(v_fn(S)); weights.append(w)
    Z, y, W = np.array(rows), np.array(targets), np.array(weights)
    y_adj = y - v_empty
    total = v_full - v_empty
    Zr = Z[:, :-1] - Z[:, -1:].reshape(-1, 1)
    yr = y_adj - Z[:, -1] * total
    WZ, Wy = Zr * np.sqrt(W)[:, None], yr * np.sqrt(W)
    beta, *_ = np.linalg.lstsq(WZ, Wy, rcond=None)
    phi = np.zeros(n); phi[:-1] = beta; phi[-1] = total - beta.sum()
    return phi


Test it!

## **Bonus : TreeSHAP (exact, model-specific) vs. KernelSHAP (sampled, model-agnostic)**

TreeSHAP exploits tree structure to compute *exact* Shapley values in polynomial time. We
compare it against KernelSHAP (which treats the tree as a black box) on both **accuracy** and
**wall-clock time**.


In [ ]:
import xgboost as xgb
from sklearn.datasets import make_regression

# Use enough features that the coalition space (2^n) is much bigger than our sample
# budgets below -- otherwise KernelSHAP silently falls back to exact enumeration and the
# budget stops mattering (try n_features=6 to see that happen!).
X_tree, y_tree = make_regression(n_samples=300, n_features=12, noise=5, random_state=0)
tree_model = xgb.XGBRegressor(n_estimators=30, max_depth=3).fit(X_tree, y_tree)

x_explain = X_tree[:5]

t0 = time.time()
tree_explainer = shap.TreeExplainer(tree_model)
sv_tree = np.array(tree_explainer.shap_values(x_explain))
t_tree = time.time() - t0

kernel_explainer = shap.KernelExplainer(tree_model.predict, X_tree[:30])

t0 = time.time()
sv_kernel_low = np.array(kernel_explainer.shap_values(x_explain, nsamples=32, silent=True))
t_kernel_low = time.time() - t0

t0 = time.time()
sv_kernel_high = np.array(kernel_explainer.shap_values(x_explain, nsamples=4000, silent=True))
t_kernel_high = time.time() - t0

print(f"TreeSHAP (exact):            {t_tree:.4f}s")
print(f"KernelSHAP, nsamples=32:     {t_kernel_low:.4f}s   | max abs error vs exact: "
      f"{np.abs(sv_tree - sv_kernel_low).max():.4f}")
print(f"KernelSHAP, nsamples=4000:   {t_kernel_high:.4f}s   | max abs error vs exact: "
      f"{np.abs(sv_tree - sv_kernel_high).max():.4f}")


**Discussion prompt:** TreeSHAP is both faster *and* exact here. What is it exploiting that
KernelSHAP can't, and under what circumstances would you still reach for KernelSHAP instead of
a model-specific method?

**Note:** even at a very large `nsamples`, KernelSHAP may not converge to *exactly* the same
values as TreeSHAP's default settings -- by default `shap.TreeExplainer` uses the
**path-dependent** perturbation convention (a different, tree-structure-specific removal
approach), while `shap.KernelExplainer` always uses the **marginal** game with an independent
background sample. This is the feature-removal axis from Module 4 showing up again: two
"exact" methods can disagree if they are silently answering two different mathematical
questions. Pass `feature_perturbation="interventional"` to `TreeExplainer` to make both use the
marginal game and see the gap close.


## **Bonus 1 : Amortized estimation: the FastSHAP idea (lightweight illustration)**

FastSHAP (Jethani et al., 2022) trains a small neural network to output Shapley value estimates
**in a single forward pass**, amortizing the cost of explanation across many inputs. Training a
full FastSHAP model (with a surrogate model, imputer, etc.) is beyond the scope of a single lab
cell, but we can illustrate the *core idea* cheaply: train a small regressor to predict
KernelSHAP's *output* directly from the input, and compare inference-time cost per example.

This is a **teaching simplification** of FastSHAP, not a faithful reimplementation --
real FastSHAP trains against the weighted least-squares objective directly rather than by
distilling a slower estimator's outputs. Treat this as a way to feel the speed/accuracy tradeoff,
not as a citable result.


In [ ]:
from sklearn.neural_network import MLPRegressor

# Generate training data: many (x, KernelSHAP(x)) pairs using the linear model from Module 3
X_train_pool = X[:100]
shap_targets = []
for xi in X_train_pool:
    sv = kernel_explainer_lin = shap.LinearExplainer(model, background).shap_values(xi.reshape(1,-1))[0]
    shap_targets.append(sv)
shap_targets = np.array(shap_targets)

amortized_explainer = MLPRegressor(hidden_layer_sizes=(32, 32), max_iter=2000, random_state=0)
amortized_explainer.fit(X_train_pool, shap_targets)

# Compare per-example inference time: amortized network vs. running KernelSHAP fresh each time
x_new = X[100:105]

t0 = time.time()
sv_amortized = amortized_explainer.predict(x_new)
t_amortized = time.time() - t0

t0 = time.time()
sv_fresh = np.array([shap.KernelExplainer(model.predict, background).shap_values(
    xi.reshape(1,-1), nsamples=200, silent=True)[0] for xi in x_new])
t_fresh = time.time() - t0

print(f"Amortized network inference time for {len(x_new)} examples: {t_amortized:.5f}s")
print(f"Fresh KernelSHAP time for {len(x_new)} examples:            {t_fresh:.5f}s")
print("Mean abs error of amortized vs fresh KernelSHAP:", np.abs(sv_amortized - sv_fresh).mean())


**Discussion prompt:** why can't we cleanly separate this amortized network into "estimate
$\hat v$" then "run $\hat\phi$ on it," the way we could for KernelSHAP in Module 3? (This is the
"fused methods" caveat from the plug-in framework discussion — FastSHAP and DASP train
end-to-end against the Shapley objective, so the two stages share parameters.)


In [ ]:
# Redefine phi-hat estimators to take an EXPLICIT rng argument, so that repeated trials
# below actually use fresh randomness each time (a common bug: capturing a fixed-seed rng
# inside a closure means every "repeat" silently replays the exact same samples!).
phi_hats_seeded = {
    "permutation (RO)": lambda v_fn, rng: phihat_permutation(v_fn, 3, 200, rng),
    "WLS regression":   lambda v_fn, rng: phihat_wls(v_fn, 3, 200, rng),
}

# Repeat several times, with a genuinely different seed each time, to see which
# (v-hat, phi-hat) combination is most STABLE under the phi-hat's own sampling noise --
# this is the "how does noise in v-hat propagate through phi-hat" question from the
# plug-in framework discussion.
n_repeats = 20
variability = {}
for v_name, v_fn in v_hats.items():
    for phi_name, phi_fn in phi_hats_seeded.items():
        trials = np.array([phi_fn(v_fn, np.random.default_rng(1000 + t))
                            for t in range(n_repeats)])
        variability[(v_name, phi_name)] = trials.std(axis=0)

print(f"{'v-hat':<26}{'phi-hat':<20}{'std(x1)':>10}{'std(x2)':>10}{'std(x3)':>10}")
for (v_name, phi_name), sd in variability.items():
    print(f"{v_name:<26}{phi_name:<20}{sd[0]:>10.4f}{sd[1]:>10.4f}{sd[2]:>10.4f}")


**What you should see:** the `mean-impute (marginal)` row has essentially **zero** variability
(both phi-hats), while the `kNN (conditional)` row has **non-zero** variability for both
phi-hats. This is not sampling noise from `phi-hat` running out of budget -- with only 3
features there are just 6 possible permutations and 6 non-trivial subsets, and a budget of 200
samples each of these many times over, so both `phi-hat` estimators have essentially converged.

**What's actually going on:** the underlying model here is exactly additive
(`3*x1 + 0*x2 + 0.5*x3`, no interaction terms), so the *marginal* game built from it is also
exactly additive -- every permutation gives literally the same marginal contribution per
feature, so there is nothing left to average over. The *conditional* game built from k-nearest-
neighbor imputation is **not** exactly additive, even though the underlying model is: which
neighbors get selected changes discretely depending on which features are "present," so the
game itself has some irreducible roughness that no amount of `phi-hat` sampling budget can
remove. This is a concrete illustration of a point Aas, Jullum & Løland (2021) and Olsen et al.
(2022) make in practice: **conditional Shapley estimators inherit variance from the v-hat
estimator itself**, not only from finite coalition sampling in `phi-hat` -- so improving `v-hat`
(a smoother conditional density estimator, more neighbors, a parametric model, etc.) can matter
more for stability than switching `phi-hat` families.

**Discussion prompt:** what would you expect to happen to the conditional row's variability if
you increased `k` (the number of neighbors) in `v_conditional`? Try it and see if your
prediction holds.


## **Bonus 3 : Benchmarking with `shapiq`**
`shapiq` (Muschalik, Fumagalli et al.) packages many of the estimators discussed in this
notebook — permutation sampling, KernelSHAP, LeverageSHAP, Owen/multilinear-extension sampling —
behind one consistent API, plus a benchmarking suite with precomputed games. This module is a
quickstart; extend it into a full accuracy-vs-budget comparison as a take-home exercise.


In [ ]:
import shapiq

# Wrap OUR toy game (from Module 1) as a shapiq.Game so we can benchmark shapiq's
# approximators against the exact values we already computed by brute force.
class ToyShapiqGame(shapiq.Game):
    def __init__(self, weights):
        self.weights = weights
        super().__init__(n_players=len(weights), normalize=False)

    def value_function(self, coalitions: np.ndarray) -> np.ndarray:
        # coalitions: array of shape (n_coalitions, n_players), boolean/0-1 membership
        out = np.zeros(coalitions.shape[0])
        for idx, S in enumerate(coalitions):
            idxs = np.where(S)[0]
            w = self.weights[idxs].sum()
            synergy = sum(0.15 * self.weights[idxs[a]] * self.weights[idxs[b]]
                          for a in range(len(idxs)) for b in range(a + 1, len(idxs)))
            out[idx] = w + synergy
        return out

shapiq_game = ToyShapiqGame(weights)  # `weights` from Module 1

budget = 300
approximators = {
    "Permutation sampling": shapiq.PermutationSamplingSV(n=n, random_state=0),
    "KernelSHAP":           shapiq.KernelSHAP(n=n, random_state=0),
    "LeverageSHAP":         shapiq.LeverageSHAP(n=n, random_state=0),
}

print(f"{'Method':<22}{'L2 error vs. exact (Module 1)':>32}")
for name, approx in approximators.items():
    result = approx.approximate(budget=budget, game=shapiq_game)
    # result.values[0] is the empty-coalition baseline; the rest are the per-player SVs
    phi_hat = result.values[1:]
    err = np.linalg.norm(phi_hat - phi_exact)
    print(f"{name:<22}{err:>32.5f}")
